# Lesson 33: LlamaIndex, LangChain + LlamaIndex, and multimodal RAG 🦙🖼️
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Build a RAG app with **LlamaIndex**: documents, nodes, index, query engine, chat engine
2. **Integrate LlamaIndex with LangChain**: use a LlamaIndex index as a LangChain retriever and as an agent tool
3. Build **multimodal RAG**: search images with text using **CLIP**, then answer with a vision model

**⏱️ Timing:** LlamaIndex 25' | integration 15' | multimodal RAG 20'

> ⚙️ **GPU recommended:** Runtime → Change runtime type → **T4 GPU** (the local LLM runs much faster).

## 🦙 LlamaIndex vs LangChain
Both are open-source frameworks for LLM apps, with different strengths:
| | **LlamaIndex** | **LangChain** |
|---|---|---|
| Focus | **Data**: loading, indexing and querying your documents | **Workflows**: chains, tools, agents, many integrations |
| Analogy | A **librarian** who organises your books | A **workshop** full of tools you connect |
| Famous for | Fast, high-quality RAG with very little code | Composable building blocks, LangGraph agents |

They work well **together**: LlamaIndex for the "find the right information" part, LangChain for the rest.

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install llama-index-core llama-index-llms-ollama llama-index-embeddings-huggingface
import warnings; warnings.filterwarnings("ignore")
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Ollama(model=MODEL, request_timeout=300, temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.chunk_size = 128; Settings.chunk_overlap = 20

In [ ]:
HANDBOOK = {
"attendance.md": """# Attendance
Students must arrive by 8:40am. Students arriving after 8:40am must sign in at the main office and collect a late slip.
If a student is sick, a parent must phone the attendance line (09 555 0101) before 9:00am or send a message through the school app.
Students leaving early must bring a signed note and sign out at the office. Unexplained absences are followed up by the dean.""",
"devices.md": """# Devices and phones
Phones must be switched off and stored in lockers during class time. Teachers may allow devices for a specific learning activity.
Phones may be used at interval and lunch in outdoor areas only. Confiscated phones can be collected from the office at 3:15pm.
Every Year 9 and 10 student needs a laptop or Chromebook with at least 6 hours of battery. The school Wi-Fi network is called StudentNet.""",
"library.md": """# Library
The library is open 8:00am to 4:30pm on weekdays and is closed on weekends and public holidays.
Students may borrow up to 5 books for three weeks. Overdue books incur no fines, but borrowing is paused until books are returned.
Quiet study rooms can be booked at the front desk for groups of up to four students. Printing costs 10 cents per page.""",
"uniform.md": """# Uniform
The school jersey must be worn from May to September. Black leather shoes are required; sports shoes are only for PE.
Hats must be worn outdoors in Terms 1 and 4 (sun-smart policy). Jewellery is limited to a watch and one pair of small studs.
Uniform items can be bought from the uniform shop, open Tuesdays and Thursdays 8:00–9:00am.""",
"assessment.md": """# Assessment
Assessment extensions must be requested from the subject teacher at least two school days before the due date.
Students who are sick on an assessment day must provide a medical certificate within five days.
Using AI tools in an assessment is only allowed when the teacher's task sheet says so, and any AI use must be acknowledged.
Plagiarism, including copying AI-generated work without acknowledgement, results in the assessment not being awarded.""",
"sport.md": """# Sport and activities
Sports trials are held in the first two weeks of each term. The senior basketball team trains Mondays and Wednesdays 3:30–5pm.
Students need a signed permission slip and must pay the activity fee before attending any trip.
The swimming sports day is in week 5 of Term 1. Togs, a towel and sunscreen are required.""",
}
import os
os.makedirs("handbook", exist_ok=True)
for name, text in HANDBOOK.items():
    open(f"handbook/{name}", "w").write(text)
print("Wrote", len(HANDBOOK), "handbook files:", list(HANDBOOK))

## Part 1: RAG in LlamaIndex — five lines 🚀
**Documents** → split into **nodes** (chunks) → **VectorStoreIndex** → **query engine**

In [ ]:
documents = SimpleDirectoryReader("handbook").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("What are the rules about phones?")
print(response, "\n")
for n in response.source_nodes:
    print(f"📄 {n.metadata['file_name']:15} score {n.score:.2f}  {n.text[:80]}…")

In [ ]:
# A chat engine remembers the conversation and rewrites follow-up questions into full questions
chat = index.as_chat_engine(chat_mode="condense_plus_context")
for msg in ["When is the library open?", "How many books can I take out from there?", "And for how long?"]:
    print(f"🧑 {msg}\n🤖 {chat.chat(msg)}\n")

In [ ]:
# Save the index so you don't have to embed everything again next time
index.storage_context.persist("handbook_index")
from llama_index.core import StorageContext, load_index_from_storage
reloaded = load_index_from_storage(StorageContext.from_defaults(persist_dir="handbook_index"))
print(reloaded.as_query_engine().query("What colour shoes do I need?"))

### 🧩 Exercise 1
1. Add a new file `handbook/canteen.md` with 3 canteen rules, rebuild the index and ask about it.
2. Change `similarity_top_k` and `Settings.chunk_size`. How do the answers and sources change?

## Part 2: LangChain + LlamaIndex together 🤝
**Pattern 1:** use LlamaIndex's retriever inside a LangChain chain.

In [ ]:
!pip -q install langchain langchain-ollama langchain-core
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

li_retriever = index.as_retriever(similarity_top_k=3)

def llamaindex_to_langchain(question):
    nodes = li_retriever.retrieve(question)                 # LlamaIndex does the finding…
    return [Document(page_content=n.text, metadata={"source": n.metadata["file_name"]}) for n in nodes]

def format_docs(docs):
    return "\n".join(f"({d.metadata['source']}) {d.page_content}" for d in docs)

chain = ({"context": RunnableLambda(llamaindex_to_langchain) | format_docs, "question": RunnablePassthrough()}
         | ChatPromptTemplate.from_template("Answer from these sources only:\n{context}\n\nQuestion: {question}")
         | ChatOllama(model=MODEL, temperature=0) | StrOutputParser())      # …LangChain does the rest
print(chain.invoke("Can I use AI in my assessment?"))

**Pattern 2:** wrap the LlamaIndex query engine as a **tool** for a LangChain agent, next to other tools. The agent decides when to look things up.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from datetime import date

@tool
def school_handbook(question: str) -> str:
    """Look up school rules: attendance, phones, library, uniform, assessment, sport."""
    return str(query_engine.query(question))

@tool
def days_until(month: int, day: int) -> int:
    """Count the days from today until a date this year (month 1-12, day 1-31)."""
    return (date(date.today().year, month, day) - date.today()).days

agent = create_agent(ChatOllama(model=MODEL, temperature=0), tools=[school_handbook, days_until],
                     system_prompt="You are a school help-desk assistant. Use the handbook tool for any school rule.")
out = agent.invoke({"messages": [{"role": "user", "content": "What time must I arrive at school, and how many days until 25 December?"}]})
for m in out["messages"]:
    print(f"{type(m).__name__:12}", getattr(m, "tool_calls", None) or m.content)

## Part 3: Multimodal RAG 🖼️🔊
**Multimodal** = more than one type of data: text, images, audio, video. **Multimodal RAG** retrieves the right **image** (or table, chart, audio clip) and gives it to a model that can **see**.

The trick: **CLIP** (OpenAI, 2021) puts images and text into the **same embedding space**, so a sentence and a matching photo end up close together.

In [ ]:
from sentence_transformers import SentenceTransformer
from skimage import data
from PIL import Image
import numpy as np, matplotlib.pyplot as plt, os

clip = SentenceTransformer("clip-ViT-B-32")
os.makedirs("images", exist_ok=True)
gallery = {"astronaut.png": data.astronaut(), "coffee.png": data.coffee(), "cat.png": data.chelsea(),
           "rocket.png": data.rocket(), "camera_man.png": np.stack([data.camera()] * 3, -1),
           "hubble.png": data.hubble_deep_field(), "retina.png": data.retina()}
for name, arr in gallery.items(): Image.fromarray(arr).save(f"images/{name}")

names = list(gallery)
image_vecs = clip.encode([Image.open(f"images/{n}") for n in names], normalize_embeddings=True)
fig, axes = plt.subplots(1, len(names), figsize=(16, 2.5))
for ax, n in zip(axes, names): ax.imshow(Image.open(f"images/{n}")); ax.set_title(n, fontsize=8); ax.axis("off")
plt.show()

In [ ]:
def image_search(text, k=1):
    q = clip.encode([text], normalize_embeddings=True)[0]
    scores = image_vecs @ q
    return [(names[i], float(scores[i])) for i in scores.argsort()[::-1][:k]]

for q in ["a pet animal", "space exploration", "a hot drink", "the night sky full of galaxies", "an eye scan"]:
    best, s = image_search(q)[0]
    print(f"{q:32} → {best:15} ({s:.2f})")

Now the **generation** step: send the retrieved image plus the question to a **vision-language model**. We'll use a small one that runs in Ollama (`moondream`, about 1.7 GB).

In [ ]:
!ollama pull moondream > /dev/null 2>&1
!pip -q install ollama
import ollama

def multimodal_rag(question):
    image_name, score = image_search(question)[0]
    display(Image.open(f"images/{image_name}").resize((200, 200)))
    r = ollama.chat(model="moondream", messages=[{"role": "user", "content": question, "images": [f"images/{image_name}"]}])
    return f"[retrieved {image_name}, similarity {score:.2f}]\n{r['message']['content']}"

print(multimodal_rag("What is the astronaut holding or wearing?"))
print(multimodal_rag("What kind of drink is in the picture, and what pattern is on top?"))

### 🧩 Exercise 2
1. Upload 5 photos of your own (classroom, sports field…) to the `images` folder, re-embed, and search them with text.
2. When does CLIP retrieve the **wrong** image? Why might that happen?
3. Real multimodal RAG often handles **PDFs with charts and tables**. How would you index a page that is half text, half chart? (Ideas: embed the text, caption the chart with a vision model and embed the caption, or embed the page image directly.)

## 📌 Summary
- **LlamaIndex**: documents → nodes → index → query/chat engine, with sources; can be saved and reloaded
- **LangChain + LlamaIndex**: LlamaIndex retrieves; LangChain chains and agents use it (as a retriever or a tool)
- **Multimodal RAG**: shared text–image embeddings (CLIP) to retrieve images, then a vision model to answer